In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [7]:
mrt = pd.read_csv('../ComputedData/MRT/full_mrt.csv')
youbike = pd.read_csv('../ComputedData/YouBike/full_youbike.csv')

In [8]:
mrt

,ExitName,PositionLat,PositionLon,City
0,哈瑪星 出入口1,22.622020,120.273400,kaoshiung
1,哈瑪星 出入口2,22.621540,120.275420,kaoshiung
2,鹽埕埔 出入口1,22.623280,120.283480,kaoshiung
3,鹽埕埔 出入口2,22.624480,120.284490,kaoshiung
4,鹽埕埔 出入口3,22.624110,120.284090,kaoshiung
...,...,...,...,...
679,出口2,24.959540,121.219320,taoyuan
680,動物園站出口,24.996173,121.576244,trtcmg
681,動物園南站出口,24.990233,121.587404,trtcmg
682,指南宮出口,24.979004,121.589748,trtcmg


In [3]:
# dataA2 = pd.read_csv("../Data/Accident/A2.csv", low_memory=False)
dataA1 = pd.read_csv("../Data/Accident/A1.csv")

In [33]:
# Step 1: 轉GeoDataFrame（經度、緯度到幾何點）投影為平面坐標系
dataA1['geometry'] = [Point(xy) for xy in zip(dataA1['經度'], dataA1['緯度'])]
mrt['geometry'] = [Point(xy) for xy in zip(mrt['PositionLon'], mrt['PositionLat'])]

gdf_data = gpd.GeoDataFrame(dataA1, geometry='geometry', crs="EPSG:4326").to_crs(epsg=3826)
gdf_mrt = gpd.GeoDataFrame(mrt, geometry='geometry', crs="EPSG:4326").to_crs(epsg=3826)

print(mrt.shape, gdf_mrt.shape)

(684, 5) (684, 5)


In [34]:
# Step 2: 建立500公尺的範圍
gdf_data['buffer'] = gdf_data.geometry.buffer(500)
gdf_buffer = gdf_data.set_geometry('buffer')

# Step 3: 空間連接（找出每個點的MRT）
joined = gpd.sjoin(gdf_data, gdf_mrt, how='left', predicate='contains')

# Step 4: 計算每個點的數量
mrt_counts = joined.groupby(joined.index).size()

print(dataA1.shape, gdf_data.shape, gdf_buffer.shape, mrt_counts.shape)

(3631, 52) (3631, 53) (3631, 53) (3631,)


In [37]:
# Step 5: 合併回原始資料表
gdf_data['mrt_500m_count'] = gdf_data.index.map(mrt_counts).fillna(0).astype(int)
gdf_data.drop(columns=['geometry', 'buffer'], inplace=True)

gdf_data.to_csv('../ComputedData/Accident/DataA1_with_MRT_counts.csv', index=False, encoding='utf-8')